Extract + Load: copy the raw csv files into DuckDb, untouched.

In [1]:
import sys
import duckdb

In [2]:
DB = "shopflow.duckdb"

In [3]:
# Parameters
DB = "shopflow_prod.duckdb"


In [4]:
TABLES = ["customers", "products", "orders", "order_items", "payments"]

In [5]:
con = duckdb.connect(DB)
con.execute("CREATE SCHEMA IF NOT EXISTS raw")

In [6]:
# query = """select
#     payment_id,
#     order_id,
#     method as payment_method,
#     created_at
# from raw.payments
# """
# n = con.execute(query).fetchall()
# print(n)
# con.close()

In [7]:
for t in TABLES:
    con.execute(f"""
                CREATE OR REPLACE TABLE raw.{t} AS
                SELECT *, CURRENT_TIMESTAMP AS _loaded_at
                FROM read_csv('data/{t}.csv', header = true)
                """)
    
    n = con.execute(f"SELECT COUNT(*) FROM raw.{t}").fetchone()[0]
    print(f"raw.{t:<12} {n:>5} rows")
con.close()

raw.customers      200 rows
raw.products         8 rows
raw.orders        1201 rows
raw.order_items   2973 rows


raw.payments      2973 rows


In [8]:
# Quick check
print(duckdb.connect('shopflow.duckdb').sql('select status, count(*) as n from raw.orders group by 1 order by 2 desc'))

┌───────────┬───────┐
│  status   │   n   │
│  varchar  │ int64 │
├───────────┼───────┤
│ DELIVERED │   216 │
│ delivered │   214 │
│ Shipped   │   203 │
│ placed    │   193 │
│ returned  │   193 │
│ shipped   │   182 │
└───────────┴───────┘



In [9]:
# Load this
# papermill load_raw.ipynb output_load_raw.ipynb p- DB "shopflow_prod.duckdb"